In [1]:
import os
from typing import List

import numpy as np
import pandas as pd

from sub_tsmd import generate

In [2]:
import scipy

def get_r(X, y, l_min, l_max):
    
    r_max = -1
    length = int(np.round((l_min + l_max) / 2))
    
    for ts, gt in zip(X, y):
        
        for (mask, motif_set) in gt:
            
            instances = np.empty(shape=(motif_set.shape[0], mask.sum(), length))
            for i, motif in enumerate(motif_set):
                c = -1
                for d in range(ts.shape[1]):
                    if not mask[d]:
                        continue
                    c += 1
                    instances[i, c, :] = scipy.signal.resample(ts[motif[0, c]:motif[1, c], d], length)      

            # Compute the distance matrix
            N, D, T = instances.shape
            distance_matrix = scipy.spatial.distance.squareform(scipy.spatial.distance.pdist(instances.reshape(N, D*T), metric='euclidean'))
            
            # Find the R for this class, and update r_max
            r = np.min(np.max(distance_matrix, axis=0))
            if r > r_max:
                r_max = r
    
    return r_max

In [3]:
def force_length(ts, gt, length, seed):
    
    # RNG generator
    rng = np.random.default_rng(seed)

    # Find the spaces that can be removed
    white_spaces = np.ones(shape=ts.shape[0], dtype=bool)
    for (_, motif_set) in gt:
        for motif in motif_set:
            for start, end in motif.T:
                white_spaces[start:end] = False
     
    # Check if it is possible to force the length
    nb_elements_to_remove = ts.shape[0] - length         
    if nb_elements_to_remove > white_spaces.sum():
        raise ValueError('Impossible to force the length!')
    
    # Select indices to remove 
    indices_to_remove = rng.choice(np.where(white_spaces)[0], nb_elements_to_remove, replace=False)
    to_remove = np.zeros(shape=ts.shape[0], dtype=bool)
    to_remove[indices_to_remove] = True
    
    # Remove the values from the time series
    ts = ts[~to_remove, :]
    
    # Remove the indices from the data
    cum_sum = np.cumsum(to_remove)
    for (_, motif_set) in gt:
        motif_set -= cum_sum[motif_set]
    
    return ts, gt

In [4]:
def generate_benchmark_set(
        start_seed: int, 
        nb_time_series: int, 
        default_length: int,
        default_dimension: int,
        lengths: List[int],
        dimensions: List[int],
        **kwargs
):
    # List for the metadata
    metadata = []
    
    # Generate the time series of different lengths
    for l in lengths:
        ds_name = f'length={l}'
        print(ds_name)
        
        # Generate the data
        ts, gt = zip(*[
            force_length(
                *generate(
                    seed=start_seed + i, 
                    dimension=default_dimension, 
                    white_space=l, 
                    **kwargs
                ),
                length=l,
                seed=start_seed + i
            )
            for i in range(nb_time_series)
        ])
                
        # Save the data
        if not os.path.exists(ds_name.lower()):
            os.mkdir(ds_name.lower())       
        pd.DataFrame(data={'ts': ts, 'gt': gt}).to_pickle(f'{ds_name}/test.pkl')

        # Save the meta data
        metadata.append(pd.Series({
            'ds_name': ds_name, 
            'D': ts[0].shape[1], 
            'l_min': kwargs['min_motif_length'], 
            'l_max': kwargs['max_motif_length'], 
            'g_max': kwargs['nb_motif_sets'], 
            'r': get_r(ts, gt, kwargs['min_motif_length'], kwargs['max_motif_length']),
            'n_avg': np.mean([i.shape[0] for i in ts])
        }))
        
    # Generate the time series of different dimensionality
    for d in dimensions:
        ds_name = f'dimension={d}'
        print(ds_name)
        
        # Generate the data
        ts, gt = zip(*[
            force_length(
                *generate(
                    seed=start_seed + i, 
                    dimension=d, 
                    white_space=default_length, 
                    **kwargs
                ),
                length=default_length,
                seed=start_seed + i
            )
            for i in range(nb_time_series)
        ])
                        
        # Save the data
        if not os.path.exists(ds_name.lower()):
            os.mkdir(ds_name.lower())       
        pd.DataFrame(data={'ts': ts, 'gt': gt}).to_pickle(f'{ds_name}/test.pkl')

        # Save the meta data
        metadata.append(pd.Series({
            'ds_name': ds_name, 
            'D': ts[0].shape[1], 
            'l_min': kwargs['min_motif_length'], 
            'l_max': kwargs['max_motif_length'], 
            'k_max': kwargs['max_motif_set_size'], 
            'r': get_r(ts, gt, kwargs['min_motif_length'], kwargs['max_motif_length']),
            'n_avg': np.mean([i.shape[0] for i in ts])
        }))
        
    # Create and store the metadata
    metadata = pd.concat(metadata, axis=1).T.set_index('ds_name')
    metadata.to_csv('metadata.csv')

In [5]:
metadata = generate_benchmark_set(
    start_seed=0,
    nb_time_series=10,
    default_length=1000,
    default_dimension=5,
    lengths=[500, 1000, 2000, 5000, 10000],
    dimensions=[3, 5, 10, 20, 30, 50, 100],
    **{
        'nb_motif_sets': 2,
        'min_motif_set_size': 2,
        'max_motif_set_size': 5,
        'min_motif_dimension': 1,
        'max_motif_dimension': 3,
        'univariate_motifs': [
            lambda x: np.sin(np.linspace(0, 2 * np.pi, x)),  # Sinusoidal 
            lambda x: np.concatenate([np.ones(int(np.ceil(x/2))), -1 * np.ones(int(np.floor(x/2)))]),  # Square wave
            lambda x: np.exp(-np.linspace(0, 2, x)) * np.sin(2 * np.pi * np.linspace(0, 1, x)),  # Damped sine
            lambda x: np.cos(np.linspace(0, 2 * np.pi, x)),  # Cosine wave
            lambda x: np.linspace(-1, 1, x),  # Linear increasings
            lambda x: np.sin(4 * np.pi * np.linspace(0, 1, x)),  # Higher frequency sine
            lambda x: np.sign(np.sin(np.linspace(0, 4 * np.pi, x))),  # Square wave-like signal
            lambda x: np.sin(4 * np.pi * np.linspace(0, 1, x)),  # Higher frequency sine
            lambda x: np.linspace(1, -1, x)  # Linear decreasing
        ], 
        'min_motif_length': 50,
        'max_motif_length': 50,
        'nb_motif_repositions': 5,
        'noise_general': 0.05,
        'noise_non_motif': 1.0,
    }
)

length=500
length=1000
length=2000
length=5000
length=10000
dimension=3
dimension=5
dimension=10
dimension=20
dimension=30
dimension=50
dimension=100
